In [14]:
import sys
from pathlib import Path
from torchvision import transforms,datasets
from torch.utils.data import Subset,DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score
sys.path.append(str(Path("..")))
from src.dataset import train_path, train_indices,val_indices,test_path

In [15]:
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

CUDA available: True
Device count: 1
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Capability: (8, 6)


In [16]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_data_aug = datasets.ImageFolder(train_path,transform=train_transform)
val_data = datasets.ImageFolder(train_path,transform=val_transform)

train_dataset_aug = Subset(train_data_aug,train_indices)
val_dataset = Subset(val_data,val_indices)

train_data_aug_loader = DataLoader(
    train_dataset_aug,
    batch_size= 32,
    shuffle= True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)


In [17]:
class CNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x

In [18]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print(device)

cuda


In [19]:
num_classes = len(train_data_aug.classes)

model = CNN(num_classes)
model = model.to(device)


In [20]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [21]:
num_epochs = 100

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
val_f1_scores = []
val_f1_scores_by_class = []


In [22]:


for epoch in range(num_epochs):


    model.train()

    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_data_aug_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_train_loss / len(train_data_aug_loader)
    train_accuracy = train_correct / train_total

    model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_val_labels.extend(labels.cpu().numpy())
            all_val_predictions.extend(predicted.cpu().numpy())

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = val_correct / val_total

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro"
    )

    f1_score_by_class = f1_score(
        all_val_labels,
        all_val_predictions,
        average= None
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    val_f1_scores.append(val_f1)
    val_f1_scores_by_class .append(f1_score_by_class)



    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )
    
    print('F1_Score by each class')
    print(f1_score_by_class)
    
    
    print('=======================================================================================')

Epoch [1/100] | Train Loss: 1.8885 | Train Acc: 0.3255 | Val Loss: 1.4574 | Val Acc: 0.4730 | Val Macro F1: 0.4115
F1_Score by each class
[0.47567568 0.49230769 0.3364486  0.22413793 0.43697479 0.54935622
 0.77714286 0.        ]
Epoch [2/100] | Train Loss: 1.2182 | Train Acc: 0.5649 | Val Loss: 1.0637 | Val Acc: 0.6246 | Val Macro F1: 0.5649
F1_Score by each class
[0.68322981 0.76576577 0.46896552 0.20952381 0.63636364 0.71153846
 0.88135593 0.16216216]
Epoch [3/100] | Train Loss: 0.8884 | Train Acc: 0.6854 | Val Loss: 0.8418 | Val Acc: 0.7162 | Val Macro F1: 0.6921
F1_Score by each class
[0.775      0.81188119 0.55454545 0.62637363 0.75       0.7431694
 0.86705202 0.40909091]
Epoch [4/100] | Train Loss: 0.6868 | Train Acc: 0.7635 | Val Loss: 0.7687 | Val Acc: 0.7252 | Val Macro F1: 0.7131
F1_Score by each class
[0.67256637 0.80769231 0.57028112 0.61375661 0.671875   0.83870968
 0.93939394 0.59016393]
Epoch [5/100] | Train Loss: 0.5492 | Train Acc: 0.8067 | Val Loss: 0.6067 | Val Acc: 

In [23]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_data = datasets.ImageFolder(
    test_path,
    transform=test_transform
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)


model.eval()

all_test_labels = []
all_test_predictions = []

test_correct = 0
test_total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_test_labels.extend(labels.cpu().numpy())
        all_test_predictions.extend(predicted.cpu().numpy())
        
test_accuracy = test_correct / test_total
test_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro"
)
test_f1_by_class = f1_score(
    all_test_labels,
    all_test_predictions,
    average=None
)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")

print("Test F1 by class:")
print(test_f1_by_class)


Test Accuracy: 0.8240
Test Macro F1: 0.8256
Test F1 by class:
[0.85714286 0.90322581 0.67961165 0.69811321 0.86597938 0.85185185
 0.97087379 0.77777778]
